# Project RxHARM — Notebook 01: HVI & HRI Computation
**Prescriptive Optimization of Heat Associated Risk Mitigation**

This notebook computes the **Heat Vulnerability Index (HVI)** and **Heat Risk Index (HRI)**
at 100-meter resolution for any city or region using freely available satellite data.

**Outputs:** GeoTIFF rasters of HVI, HRI, and all 14 sub-indicators saved to Google Drive.

📄 Paper DOI: `https://doi.org/PLACEHOLDER`  
💻 GitHub: `https://github.com/YOUR_USERNAME/rxharm`

> **Before you begin:** Run cells in order. Each cell must complete before running the next.
> Cells 1–3 are setup; Cell 5 is the only cell you need to edit.

In [ ]:
# ── Cell 2: Installation ────────────────────────────────────────────────────
# Install the RxHARM package directly from GitHub.
# Run this cell once per Colab session (takes ~2-3 minutes on first run).

!pip install -q git+https://github.com/YOUR_USERNAME/rxharm.git

# Verify installation
import rxharm
print(f'RxHARM version: {rxharm.__version__} installed successfully')

In [ ]:
# ── Cell 3: Authentication ──────────────────────────────────────────────────
# Authenticate Google Earth Engine.
# You will be prompted to log in with your Google account.

import ee
ee.Authenticate()
ee.Initialize(project='YOUR_GEE_PROJECT_ID')  # <-- replace with your GEE project ID

# Mount Google Drive for saving outputs
from google.colab import drive
drive.mount('/content/drive')

print('Authentication complete.')

## USER INPUTS — Edit the values in the cell below

This is the **only cell you need to edit.** Change the values to match your study area.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# USER INPUTS — Only edit values in this cell
# ════════════════════════════════════════════════════════════════════

# Location: city name, path to shapefile, or (lat, lon, radius_km)
LOCATION = 'Ahmedabad, India'

# Analysis year (2016-2024 supported for full Tier 1 indicator stack)
YEAR = 2023

# Weighting method for indicator aggregation within each sub-index
# Options: 'equal' | 'pca' | 'entropy' | 'critic' | 'manual'
WEIGHTING_METHOD = 'equal'

# Google Drive folder for saving outputs
DRIVE_OUTPUT_FOLDER = 'RxHARM_outputs'

# Whether to export rasters to Drive (True) or preview only (False)
EXPORT_TO_DRIVE = True

# ════════════════════════════════════════════════════════════════════
# Step II now functional: AOI preview below ↓
# ════════════════════════════════════════════════════════════════════
print(f'Location: {LOCATION} | Year: {YEAR} | Weighting: {WEIGHTING_METHOD}')

In [ ]:
# ── AOI Processing (Step II — now working) ──────────────────────────────────
from rxharm.aoi.handler import AOIHandler
from rxharm.aoi.decomposer import ZoneDecomposer

print('Processing AOI...')
aoi = AOIHandler(LOCATION, YEAR)
aoi.validate()
aoi.display_summary()

print('\nPreparing zone structure...')
decomposer = ZoneDecomposer(aoi)
zones = decomposer.decompose()
print(f"Ready: {zones['n_zones']} zones in '{zones['mode']}' mode")
print(f'Köppen climate zone: {aoi.get_koppen_zone()}')
print(f'\nEstimated total runtime: {aoi.estimate_runtime()}')
print('\n[Step III will add: GEE indicator fetching and export]')

In [ ]:
# ── Step 1: Process AOI ─────────────────────────────────────────────────────
# Will be implemented in Step II of the build process.
# When complete, this cell will:
#   - Geocode or load your AOI geometry
#   - Classify AOI size (moore/direct/meso/hierarchical)
#   - Display an interactive preview map
#   - Estimate total runtime

print('Step 1: AOI Processing')
print(f'  Location  : {LOCATION}')
print(f'  Year      : {YEAR}')
print('  Status    : AOI processing available after Step II is complete.')
print()
print('The full pipeline will be available in later build steps.')
print('See the project README for the build roadmap.')

In [ ]:
# ── Step 2: Seasonal Detection ──────────────────────────────────────────────
# Will be implemented in Step III.
# When complete, queries ERA5 to identify the hottest calendar months
# for use as the Landsat and Sentinel-2 composite window.

print('Step 2: Seasonal Detection')
print('  Status: Seasonal detection available after Step III is complete.')
print('  Output will be: hottest_months (list of month integers, e.g. [5, 6])')

In [ ]:
# ── Step 3: Fetch All 14 Indicators from GEE ────────────────────────────────
# Will be implemented in Step III.
# When complete, fetches and mosaics:
#   Hazard   : LST, Albedo, UHI (Landsat C2)
#   Exposure : Population, Built Fraction (WorldPop, GHS-BUILT-S)
#   Sensitivity: Elderly/Child fractions, Impervious, Cropland
#   AC       : NDVI, NDWI, Tree Cover, Canopy Height, VIIRS-DNB

print('Step 3: GEE Data Fetch (14 Indicators)')
print('  Status: Data fetching available after Step III is complete.')
print('  Expected runtime when implemented: ~15-40 min depending on AOI size.')

In [ ]:
# ── Step 4: Compute HVI and HRI ─────────────────────────────────────────────
# Will be implemented in Step IV.
# When complete, applies:
#   HVI = (E x S) / AC    (all sub-indices normalized 0-1)
#   HRI = H_s x HVI       (spatial hazard x vulnerability)

print('Step 4: HVI and HRI Computation')
print('  Status: Index computation available after Step IV is complete.')
print()
# Preview config to confirm weights are loaded correctly
cfg = rxharm.load_config()
print(f'  Hazard weights    : {cfg.HAZARD_WEIGHTS}')
print(f'  Exposure weights  : {cfg.EXPOSURE_WEIGHTS}')
print(f'  Sensitivity weights: {cfg.SENSITIVITY_WEIGHTS}')
print(f'  AC weights        : {cfg.ADAPTIVE_CAPACITY_WEIGHTS}')

In [ ]:
# ── Step 5: Export Results ───────────────────────────────────────────────────
# Will be implemented in Step VI.
# When complete, writes HVI/HRI rasters to Google Drive.

print('Step 5: Export Results')
print(f'  Export target : /content/drive/MyDrive/{DRIVE_OUTPUT_FOLDER}/')
print(f'  Export enabled: {EXPORT_TO_DRIVE}')
print('  Status: Export available after Step VI is complete.')
print()
print('Notebook 01 skeleton is ready. Build Steps II-VI will wire up all cells.')

## Step 3: Fetch All 14 Indicators

This step queries Google Earth Engine to retrieve all HVI indicator data
for your location and year. The export runs asynchronously — you will
be given a link to monitor progress.

**Estimated time:** 5–15 minutes for the GEE export to complete.

In [ ]:
# ── Authenticate GEE (if not already done) ────────────────────────────────
import ee
try:
    ee.data.getAlgorithms()
    print('GEE already authenticated')
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)

# Detect hottest period from ERA5 climatology
from rxharm.seasonal.detector import SeasonalDetector
from tqdm.notebook import tqdm

print('Detecting hottest period from ERA5 climatology...')
detector = SeasonalDetector(aoi.centroid_ll[0], aoi.centroid_ll[1], YEAR)
hottest_months = detector.detect(use_cache=True)
print(f'Hottest months for {LOCATION}: {hottest_months}')
print(f'MMT estimate: {detector.get_climatological_mmt():.1f}°C')

In [ ]:
# ── Fetch all 14 indicators ────────────────────────────────────────────────
from rxharm.fetch import fetch_all_indicators

print('Starting GEE data fetch for all 14 indicators...')
print('(This submits an export job — the notebook can continue while it runs)\n')

fetch_result = fetch_all_indicators(
    aoi_handler=aoi,
    seasonal_detector=detector,
    export_to_drive=EXPORT_TO_DRIVE,
    drive_folder=DRIVE_OUTPUT_FOLDER,
    show_progress=True,
)

print(f"\nIndicators fetched: {fetch_result['band_names']}")
print(f"Total bands: {len(fetch_result['band_names'])}")

if fetch_result['export_task']:
    task = fetch_result['export_task']
    print(f"\nExport task status: {task.status()['state']}")
    print('Check: https://code.earthengine.google.com/tasks')

In [ ]:
# ── Run this cell AFTER the GEE export has finished ────────────────────────
# Load the exported GeoTIFF and run VIIRS downscaling

import rasterio
import numpy as np
import glob
from rxharm.fetch.viirs_downscaler import VIIRSDownscaler

EXPORTED_TIFF = f'/content/drive/MyDrive/{DRIVE_OUTPUT_FOLDER}/RxHARM_indicators_{YEAR}_*.tif'
tiff_files = glob.glob(EXPORTED_TIFF)

if not tiff_files:
    print('GeoTIFF not found. Make sure the export completed.')
    print(f'Expected: /content/drive/MyDrive/{DRIVE_OUTPUT_FOLDER}/')
else:
    tiff_path = tiff_files[-1]
    print(f'Loading: {tiff_path}')

    with rasterio.open(tiff_path) as src:
        band_names = fetch_result['band_names']
        def band_idx(name): return band_names.index(name) + 1
        viirs_raw  = src.read(band_idx('viirs_dnb_raw'))
        population = src.read(band_idx('population'))
        ndvi       = src.read(band_idx('ndvi'))
        built_frac = src.read(band_idx('built_frac'))

    downscaler = VIIRSDownscaler(n_estimators=100, random_state=42)
    print('Running RFATPK downscaling (1–3 minutes)...')

    viirs_100m = downscaler.downscale(
        viirs_coarse=viirs_raw,
        covariates_fine={
            'population': population,
            'ndvi': ndvi,
            'built_frac': built_frac,
        },
    )

    print(f'Downscaling complete. Output shape: {viirs_100m.shape}')
    print(f'Value range: [{viirs_100m.min():.2f}, {viirs_100m.max():.2f}] nW/cm\u00b2/sr')
    print('\n[Ready for Step IV: HVI computation]')